# Download, Host, and Query Your Own AI Model

## Classroom learning goals 

By the end of this notebook, you will be able to:

- Explain the difference between downloading a model and loading it into memory.
- Identify the tokenizer and the model weights, and describe what each part does.
- Check whether the model will run on a CPU or a GPU.
- Send prompts to a locally hosted model and inspect its responses.
- Describe the limits of a base language model.

## What will happen?

1. **Install dependencies:** Add Python libraries that can download and run the model.
2. **Choose storage:** Create a local folder where downloaded model files can be cached.
3. **Check hardware:** Determine whether a GPU is available; otherwise, use the CPU.
4. **Download and load:** Retrieve the model and tokenizer, then move the model into memory.
5. **Test the model:** Ask several prepared questions and measure generation time.
6. **Explore interactively:** Enter your own prompts in a text box.

> **Important:** In this notebook, “hosting” means running the model inside the current Jupyter process. It is not a public web service or API.

---

In [1]:
# STEP 1: Install the Python libraries used by this notebook.
#
# - transformers: downloads and runs the model and tokenizer
# - torch: provides the numerical runtime used for AI calculations
# - accelerate: helps Transformers manage model loading
# - ipywidgets: creates the interactive text box and button
#
# %pip uses the Jupyter kernel's Python environment, unlike a plain system pip command.
%pip install --quiet transformers ipywidgets torch accelerate

print("Required libraries are installed.")

Note: you may need to restart the kernel to use updated packages.
Required libraries are installed.


In [2]:
# STEP 2: Create a folder for cached model files.
from pathlib import Path

# The cache belongs to the current user and is independent of the notebook directory.
MODELS_DIR = Path.home() / "ai_models"

# mkdir(..., exist_ok=True) creates the folder if needed and does not fail if it exists.
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model cache directory: {MODELS_DIR}")
print("This folder may contain downloaded tokenizer files and model weights.")

Model cache directory: /home/jovyan/ai_models
This folder may contain downloaded tokenizer files and model weights.


## Step 3: Load the Base Language Model

A language model needs computational resources to generate text. A GPU is typically faster, but a CPU is sufficient for this beginner demonstration.

### Two important objects

- **Tokenizer:** Converts text into numerical token IDs and converts generated IDs back into readable text.
- **Model:** Contains learned weights that estimate which tokens should follow a prompt.

This notebook uses `google/flan-t5-base`, a sequence-to-sequence model that is relatively small and works for introductory demonstrations. Depending on available memory, it may take several minutes to download on the first run.

In [3]:
# STEP 3A: Detect available hardware.
#
# CUDA indicates an NVIDIA GPU is available through PyTorch.
# If CUDA is unavailable, the notebook uses the CPU instead.
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("CUDA available:", torch.cuda.is_available())
print("Selected device:", DEVICE.upper())

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print("GPU:", gpu_name)
    print("GPU memory:", round(gpu_memory_gb, 1), "GB")
else:
    print("The CPU will be used. Generation may be slower.")

CUDA available: False
Selected device: CPU
The CPU will be used. Generation may be slower.


In [4]:
# STEP 3B: Download and load the tokenizer and model.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# The model identifier tells Transformers which model files to retrieve from the Hub.
model_name = "google/flan-t5-base"

# For a slower or memory-limited CPU environment, instructors may replace the line
# above with: model_name = "google/flan-t5-small"

print(f"Model: {model_name}")
print(f"Cache directory: {MODELS_DIR}")
print("Step 1 of 2: downloading/loading the tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    cache_dir=MODELS_DIR,
)

print("Step 2 of 2: downloading/loading the model weights...")
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    cache_dir=MODELS_DIR,
).to(DEVICE)

print("The model and tokenizer are now hosted by this notebook process.")
print("They are ready to receive prompts.")

Model: google/flan-t5-base
Cache directory: /home/jovyan/ai_models
Step 1 of 2: downloading/loading the tokenizer...
Step 2 of 2: downloading/loading the model weights...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


The model and tokenizer are now hosted by this notebook process.
They are ready to receive prompts.


In [5]:
# Confirm device running model - GPU/CUDA or CPU
print(f"Model running on : {next(model.parameters()).device}")

Model running on : cpu


## Query the hosted model

In [6]:
questions = [
    "What are the main components of the Solar System?",
    "How does the Internet work at a high level?",
    "What are some applications of artificial intelligence?",
    "Explain what makes a large language model useful.",
]

In [7]:
def ask(model, tokenizer, question, max_new_tokens=200):
    """Send a prompt directly to the locally hosted model."""
    inputs = tokenizer(question, return_tensors="pt").to(DEVICE)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_beams=4,
        no_repeat_ngram_size=2,
    )
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("Q:", question)
    print("A:", answer)
    print("-" * 80)
    return answer

## Run sample questions

The questions below let us verify that the downloaded model is loaded and generating answers. Total runtime is shown at the end.

In [8]:
%%time

for question in questions:
    ask(model, tokenizer, question)

Q: What are the main components of the Solar System?
A: atoms
--------------------------------------------------------------------------------
Q: How does the Internet work at a high level?
A: The Internet works at a high level by transferring data from one computer to another computer.
--------------------------------------------------------------------------------
Q: What are some applications of artificial intelligence?
A: Artificial Intelligence
--------------------------------------------------------------------------------
Q: Explain what makes a large language model useful.
A: A large language model can be used to describe a wide range of languages.
--------------------------------------------------------------------------------
CPU times: user 13.9 s, sys: 331 ms, total: 14.2 s
Wall time: 7.46 s


## Step 6: Use the Interactive Prompt

This final interface lets each student type an original question and receive a response from the model running in this notebook.

### How the interface works

- `Text` creates the question input box.
- `Button` creates the button that submits the prompt.
- `Output` is a dedicated area where the response appears.
- `submit_question()` reads the input, clears the previous response, and calls `ask()`.
- Pressing **Enter** or clicking **Ask model** starts generation.

### Classroom activities

1. Ask one straightforward factual question.
2. Ask a follow-up with a different wording.
3. Ask the model to explain a concept at a beginner level.
4. Compare the response with a known source.
5. Ask an ambiguous question and discuss how the model handles missing context.

> **Model limitation:** This notebook uses a base model from the Transformers library, not a live Internet-connected assistant. It cannot browse the web or verify facts unless they were learned during model training.

In [10]:
# Interactive prompt
# Enter a question in the text box and the locally hosted model will respond.
import ipywidgets as widgets
from IPython.display import display

question_box = widgets.Text(
    placeholder="Ask the local model a question...",
    description="Question:",
    layout=widgets.Layout(width="80%"),
)
ask_button = widgets.Button(description="Ask model", button_style="primary")
output = widgets.Output()

def submit_question(_):
    question = question_box.value.strip()
    if not question:
        return

    with output:
        output.clear_output(wait=True)
        ask(model, tokenizer, question, max_new_tokens=300)

ask_button.on_click(submit_question)
# question_box.on_submit(submit_question)

display(question_box, ask_button, output)

print("Tip: press Enter or click 'Ask model' to submit a question.")

Text(value='', description='Question:', layout=Layout(width='80%'), placeholder='Ask the local model a questio…

Button(button_style='primary', description='Ask model', style=ButtonStyle())

Output()

Tip: press Enter or click 'Ask model' to submit a question.
